In [0]:
import sys
sys.path.append("/Workspace/Repos/juliano.garciagregorio@gmail.com/mentoria-dados/data-product-crypto/src")
import zordon

In [0]:
import requests
import json


In [0]:
import zordon

proj = zordon.Project(spark, country="br", region="sa", environment="dev")

bronze_binance = proj.client(layer="bronze", domain="binance", subdomain="ohlcv")

In [0]:
import requests
import time
from datetime import datetime, timezone

def fetch_klines(symbol, interval, start_ts=None, limit=1000):
    url = "https://data-api.binance.vision/api/v3/klines"
    params = {"symbol": symbol, "interval": interval, "limit": limit}
    if start_ts:
        params["startTime"] = start_ts

    for tentativa in range(5):
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            time.sleep(2 ** tentativa)
        else:
            resp.raise_for_status()
    raise Exception("Falha após múltiplas tentativas de fetch")

In [0]:
symbol = "BTCUSDT"
interval = "1h"

try:
    df_existente = bronze_binance.read_table(
        "klines",
        filters={"symbol": symbol, "interval": interval}
    )
    last_open_time = df_existente.agg({"open_time": "max"}).collect()[0][0]
    start_ts = int(last_open_time) + 1 if last_open_time else None
except Exception:
    # tabela ainda não existe -> primeira carga (backfill completo)
    start_ts = None

In [0]:
raw_data = fetch_klines(symbol, interval, start_ts=start_ts)
ingestion_ts = datetime.now(timezone.utc).isoformat()

if not raw_data:
    print(f"[{ingestion_ts}] Nenhum candle novo para {symbol}/{interval} (start_ts={start_ts}).")
else:
    registros = [
        {
            "symbol": symbol,
            "interval": interval,
            "open_time": candle[0],
            "open": candle[1],
            "high": candle[2],
            "low": candle[3],
            "close": candle[4],
            "volume": candle[5],
            "close_time": candle[6],
            "quote_asset_volume": candle[7],
            "num_trades": candle[8],
            "taker_buy_base_vol": candle[9],
            "taker_buy_quote_vol": candle[10],
            "ignore": candle[11],
            "ingestion_timestamp": ingestion_ts
        }
        for candle in raw_data
    ]

    df_novo = spark.createDataFrame(registros)

    fqn = bronze_binance.upsert_table(
        df_novo,
        "klines",
        merge_keys=["symbol", "interval", "open_time"],
        partition_cols=["symbol"]
    )
    print(f"[{ingestion_ts}] {len(registros)} candles processados em {fqn}")